# 01 — Why MLOps?

This notebook introduces the main challenges encountered when moving machine learning models from experimentation to production.

## Learning objectives

By the end of this notebook, you will be able to:

- Understand the AI adoption gap.
- Explain why reproducibility matters.
- Simulate a simple data drift scenario.
- Identify technical debt in ML projects.
- Map production failures to MLOps solutions.

## 1. Setup

We use only standard data science libraries: `numpy`, `pandas`, `matplotlib`, and `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. The AI Adoption Gap

Many ML projects start as promising prototypes but only a small fraction reach reliable production deployment.

The following example simulates a typical project funnel.

In [ ]:
projects = pd.DataFrame({
    "stage": ["Idea", "Prototype", "Validated model", "Pilot deployment", "Production"],
    "count": [100, 45, 25, 12, 5]
})

projects

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(projects["stage"], projects["count"])
plt.title("Illustrative AI Adoption Gap")
plt.xlabel("Project stage")
plt.ylabel("Number of projects")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Discussion

The drop between prototype and production is rarely caused by model performance alone.

Typical blockers include:

- Missing deployment process.
- Lack of reproducibility.
- No monitoring.
- Weak governance.
- Poor collaboration between teams.

## 3. Reproducibility Problem

A model result should be reproducible.

This requires tracking:

- Data version.
- Code version.
- Parameters.
- Random seed.
- Execution environment.

In [ ]:
# Experiment A
np.random.seed(42)
sample_a = np.random.normal(loc=0, scale=1, size=1000)
sample_a.mean()

In [ ]:
# Experiment B
np.random.seed(123)
sample_b = np.random.normal(loc=0, scale=1, size=1000)
sample_b.mean()

Even this simple example produces different results when the random seed changes.

In real ML systems, reproducibility is affected by many additional factors:

- Dataset extraction date.
- Feature engineering logic.
- Library versions.
- Model hyperparameters.
- Hardware and runtime environment.

## 4. Dataset Versioning Problem

A model trained on one dataset version may behave differently from a model trained on another version.

The following example illustrates why dataset versioning matters.

In [ ]:
customers_v1 = pd.DataFrame({
    "income": [30000, 50000, 80000],
    "age": [22, 35, 48],
    "target": [0, 0, 1]
})

customers_v2 = pd.DataFrame({
    "income": [30000, 50000, 80000, 100000],
    "age": [22, 35, 48, 60],
    "target": [0, 0, 1, 1]
})

customers_v1

In [ ]:
customers_v2

### Questions

1. Which dataset version was used to train the model?
2. Were records added, removed, or corrected?
3. Can the same dataset be reconstructed later?
4. Which controls should be implemented?

## 5. Data Drift Example

Data drift occurs when the statistical properties of production data differ from the training data.

We simulate a binary classification problem and then create a shifted production population.

In [ ]:
X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    class_sep=1.5,
    random_state=RANDOM_SEED
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

test_pred_proba = model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_pred_proba)

test_auc

In [ ]:
# Create production data with shifted feature values
X_prod = X_test.copy()
X_prod[:, 0] = X_prod[:, 0] + 2.0
X_prod[:, 1] = X_prod[:, 1] - 1.5

prod_pred_proba = model.predict_proba(X_prod)[:, 1]
prod_auc = roc_auc_score(y_test, prod_pred_proba)

pd.DataFrame({
    "dataset": ["Test data", "Shifted production data"],
    "auc": [test_auc, prod_auc]
})

In [ ]:
comparison = pd.DataFrame({
    "feature": ["feature_0", "feature_1"],
    "training_mean": [X_test[:, 0].mean(), X_test[:, 1].mean()],
    "production_mean": [X_prod[:, 0].mean(), X_prod[:, 1].mean()]
})

comparison

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(X_test[:, 0], bins=30, alpha=0.6, label="Test data")
plt.hist(X_prod[:, 0], bins=30, alpha=0.6, label="Production data")
plt.title("Example of Data Drift on Feature 0")
plt.xlabel("Feature value")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

### Interpretation

The input distribution changed between test and production data.

In production, such changes may be caused by:

- New customer populations.
- Market conditions.
- Product changes.
- Data collection changes.
- External shocks.

## 6. Concept Drift Example

Concept drift occurs when the relationship between input variables and the target changes.

In credit scoring, this may happen during a recession, when variables that previously predicted low risk may become less reliable.

In [ ]:
# Original relationship
income = np.random.normal(50000, 12000, 1000)
debt_ratio = np.random.uniform(0.1, 0.8, 1000)

default_risk_original = 1 / (1 + np.exp(-(3 * debt_ratio - 0.00003 * income)))
default_original = np.random.binomial(1, default_risk_original)

# New relationship after economic shock
default_risk_new = 1 / (1 + np.exp(-(4.5 * debt_ratio - 0.000015 * income + 0.5)))
default_new = np.random.binomial(1, default_risk_new)

concept_df = pd.DataFrame({
    "income": income,
    "debt_ratio": debt_ratio,
    "default_original": default_original,
    "default_new": default_new
})

concept_df.head()

In [ ]:
concept_summary = pd.DataFrame({
    "period": ["Original environment", "New environment"],
    "default_rate": [
        concept_df["default_original"].mean(),
        concept_df["default_new"].mean()
    ]
})

concept_summary

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(concept_summary["period"], concept_summary["default_rate"])
plt.title("Concept Drift: Default Rate Change")
plt.ylabel("Default rate")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 7. Technical Debt in ML Systems

Technical debt appears when short-term implementation choices create long-term maintenance costs.

Examples:

- Hard-coded thresholds.
- Undocumented feature transformations.
- Manual retraining.
- Duplicate scripts.
- No tests.

In [ ]:
# Example of fragile, hard-coded logic

def manual_score(customer):
    if customer["income"] > 50000:
        if customer["age"] > 25:
            return 1
        else:
            return 0
    else:
        return 0

manual_score({"income": 60000, "age": 30})

This function is simple but difficult to govern:

- No versioning.
- No statistical validation.
- No monitoring.
- No documentation of assumptions.
- No separation between business rules and model logic.

## 8. Failed Credit Scoring Case Study

A bank developed a credit scoring model.

Initial validation performance:

```text
AUC = 0.84
```

After 12 months in production:

```text
AUC = 0.66
```

Observed problems:

- No monitoring.
- No retraining process.
- Missing experiment tracking.
- Dataset not versioned.
- Feature engineering undocumented.

In [ ]:
failures = pd.DataFrame({
    "problem": [
        "Performance degradation",
        "Unknown dataset version",
        "No retraining process",
        "Missing documentation",
        "Untracked experiments"
    ],
    "mlops_area": [
        "Monitoring",
        "Versioning",
        "Automation",
        "Governance",
        "Reproducibility"
    ],
    "corrective_action": [
        "Implement model performance monitoring",
        "Introduce dataset versioning",
        "Create automated retraining pipeline",
        "Create model documentation standards",
        "Use experiment tracking such as MLflow"
    ]
})

failures

## 9. MLOps Solutions Mapping

In [ ]:
solutions = pd.DataFrame({
    "production_problem": [
        "Lost source code",
        "Lost dataset",
        "Untracked experiments",
        "Environment mismatch",
        "Manual deployment",
        "Model degradation"
    ],
    "mlops_solution": [
        "Git / GitHub",
        "DVC / Delta Lake / LakeFS",
        "MLflow Tracking",
        "Docker",
        "CI/CD pipeline",
        "Monitoring and drift detection"
    ]
})

solutions

## 10. Exercises

### Exercise 1 — Reproducibility

Change the random seed in the reproducibility section and observe the impact.

Questions:

1. Why do results change?
2. Which metadata should be stored?
3. How would you make the experiment reproducible?

### Exercise 2 — Dataset Versioning

Compare `customers_v1` and `customers_v2`.

Questions:

1. Which records changed?
2. What should be tracked?
3. Which tools could help?

### Exercise 3 — Drift Analysis

Modify the shift applied to `X_prod`.

Questions:

1. How does the AUC change?
2. Which features are most affected?
3. What kind of alert would you define?

### Exercise 4 — Failed ML Project

Using the failure table, propose a corrective action plan for the bank.

## 11. Key Takeaways

- Building a model is only one step in the ML lifecycle.
- Reproducibility is essential for auditability and trust.
- Data drift and concept drift can reduce model performance over time.
- Technical debt creates operational risk.
- MLOps provides practices and tools to move from experimentation to reliable production systems.